<a href="https://colab.research.google.com/github/FrankAlvaradoR/c_plusplus/blob/main/Practica_3_pograAvanza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%%writefile calentamiento.cpp
#include <stdio.h>

// 1. MACRO SIMPLE: Definimos una constante global de preprocesador
#define UMBRAL_VOLTAJE 5

// 2. FUNCIÓN SIMPLE: Evalúa si un voltaje digital requiere alerta
void verificar_voltaje(int voltaje_leido) {
    if (voltaje_leido >= UMBRAL_VOLTAJE) {
        printf("[ESTADO]: Voltaje seguro (%d V). Sistema operando.\n", voltaje_leido);
    } else {
        printf("[ALERTA]: Voltaje bajo (%d V). Revisar fuente!\n", voltaje_leido);
    }
}

int main() {
    printf("--- CALENTAMIENTO: SENSORES Y MACROS ---\n\n");

    // Simulamos lecturas de un puerto de hardware
    int lectura_sensor_1 = 6;
    int lectura_sensor_2 = 3;

    // Invocamos la función simple pasando las lecturas
    verificar_voltaje(lectura_sensor_1);
    verificar_voltaje(lectura_sensor_2);

    return 0;
}

Writing calentamiento.cpp


In [6]:
!gcc calentamiento.cpp -o calentamiento
!./calentamiento

--- CALENTAMIENTO: SENSORES Y MACROS ---

[ESTADO]: Voltaje seguro (6 V). Sistema operando.
[ALERTA]: Voltaje bajo (3 V). Revisar fuente!


In [7]:
%%writefile intermedio_bits.cpp
#include <stdio.h>

int main() {
    printf("--- TALLER INTERMEDIO: ENMASCARAMIENTO DE SENsores ---\n\n");

    // Simulamos un registro de 8 bits que proviene del hardware del microcontrolador.
    // Supongamos que el valor actual en binario es 0xB4 (1011 0100 en binario)
    // Distribución lógica simulada:
    // Bits [0-3]: Sensores de temperatura (4 canales)
    // Bits [4-6]: Estado de flags de error (Voltaje, Memoria, Comunicación)
    // Bit  [7]  : Interruptor general de energía (1 = Encendido, 0 = Apagado)
    unsigned char registro_hw = 0xB4;

    // 1. IMPRESIÓN DEL ESTADO ORIGINAL EN BINARIO
    printf("Estado crudo del hardware (Hex: 0x%X):\nBinario: ", registro_hw);
    for (int i = 7; i >= 0; i--) {
        printf("%d", (registro_hw >> i) & 1);
        if (i == 4) printf(" ");
    }
    printf("\n\n");

    // =========================================================================
    // RETO 1: Aislar los sensores de temperatura (Bits 0 al 3)
    // Para ello, necesitamos una máscara que deje pasar solo los primeros 4 bits.
    // La máscara en hexadecimal es 0x0F (en binario: 0000 1111).
    // =========================================================================
    unsigned char mascara_temperatura = 0x0F;
    unsigned char lectura_temperaturas = registro_hw & mascara_temperatura;

    printf(">> Reto 1: Leyendo canales de temperatura (Bits 0-3)...\n");
    printf("   Valor aislado (Hex): 0x%X\n\n", lectura_temperaturas);

    // =========================================================================
    // RETO 2: Comprobar si el interruptor general de energía (Bit 7) está activo.
    // Desplazamos el bit 7 hasta la posición 0 y aplicamos un AND con 1.
    // =========================================================================
    int interruptor_general = (registro_hw >> 7) & 1;

    printf(">> Reto 2: Comprobando interruptor de energia (Bit 7)...\n");
    if (interruptor_general == 1) {
        printf("   [ESTADO]: El sistema principal esta ENCENDIDO.\n\n");
    } else {
        printf("   [ESTADO]: El sistema principal esta APAGADO.\n\n");
    }

    // =========================================================================
    // RETO 3: Aislar las flags de error (Bits 4, 5 y 6) sin alterar los demás.
    // Creamos una máscara para los bits 4, 5 y 6 (0x70 en Hex -> 0111 0000 en Binario)
    // =========================================================================
    unsigned char mascara_errores = 0x70;
    unsigned char estado_errores = (registro_hw & mascara_errores) >> 4; // Los alineamos al inicio

    printf(">> Reto 3: Analizando bloque de errores (Bits 4-6)...\n");
    printf("   Codigo de error detectado (desplazado): %d\n", estado_errores);

    return 0;
}

Writing intermedio_bits.cpp


In [8]:
!gcc intermedio_bits.cpp -o intermedio
!./intermedio

--- TALLER INTERMEDIO: ENMASCARAMIENTO DE SENsores ---

Estado crudo del hardware (Hex: 0xB4):
Binario: 1011 0100

>> Reto 1: Leyendo canales de temperatura (Bits 0-3)...
   Valor aislado (Hex): 0x4

>> Reto 2: Comprobando interruptor de energia (Bit 7)...
   [ESTADO]: El sistema principal esta ENCENDIDO.

>> Reto 3: Analizando bloque de errores (Bits 4-6)...
   Codigo de error detectado (desplazado): 3


### **Código Practica 3**

In [3]:
%%writefile practica3.cpp
#include <stdio.h>

// =========================================================================
// 1. Declaración de Rutinas Específicas (Callbacks)
// =========================================================================
void leer_voltaje(void) {
    printf("[ SISTEMA ]: Leyendo voltaje del bus de sensores... Valor actual: 12.4V\n");
}

void ejecutar_calibracion(void) {
    printf("[ SISTEMA ]: Iniciando rutina de calibracion de circuiteria...\n");
    printf("[ SISTEMA ]: Calibracion completada exitosamente.\n");
}

void activar_alarma(void) {
    printf("[ ALERTA ]: Activando protocolo de emergencia industrial!\n");
    printf("[ ALERTA ]: Sirenas y balizas encendidas.\n");
}

int main() {
    printf("--- CONSOLA INDUSTRIAL - ARQUITECTURA MODULAR ---\n\n");

    // =========================================================================
    // 2. Implementación de la Tabla de Despacho (Apuntadores a Funciones)
    // =========================================================================
    // Declaramos un arreglo de apuntadores a funciones que reciben 'void' y devuelven 'void'
    void (*menu_acciones[])(void) = {
        leer_voltaje,          // Índice 0
        ejecutar_calibracion,  // Índice 1
        activar_alarma         // Índice 2
    };

    // Simulamos la elección del usuario (en un menú interactivo real se usaría scanf)
    // Probaremos la ejecución secuencial directa de las opciones de la tabla:
    int opcion_simulada = 1; // Seleccionamos la opción 1 (Calibración)

    printf("Simulando seleccion de menu del usuario (Opcion: %d)...\n", opcion_simulada + 1);

    // =========================================================================
    // 3. Invocación Dinámica Mediante Callback
    // =========================================================================
    // Validamos el rango antes de invocar la función de forma dinámica
    if (opcion_simulada >= 0 && opcion_simulada < 3) {
        // Invocamos la función directamente usando el apuntador almacenado en el arreglo
        menu_acciones[opcion_simulada]();
    } else {
        printf("[ ERROR ]: Opcion de menu invalida.\n");
    }

    // Probemos otra opción de la tabla (Opción 2: Alarma)
    opcion_simulada = 2;
    printf("\nSimulando seleccion de menu del usuario (Opcion: %d)...\n", opcion_simulada + 1);

    if (opcion_simulada >= 0 && opcion_simulada < 3) {
        menu_acciones[opcion_simulada]();
    }

    printf("\n--- FIN DE LA SIMULACION ---\n");
    return 0;
}

Overwriting practica3.cpp


In [4]:
!gcc practica3.cpp -o practica3
!./practica3

--- CONSOLA INDUSTRIAL - ARQUITECTURA MODULAR ---

Simulando seleccion de menu del usuario (Opcion: 2)...
[ SISTEMA ]: Iniciando rutina de calibracion de circuiteria...
[ SISTEMA ]: Calibracion completada exitosamente.

Simulando seleccion de menu del usuario (Opcion: 3)...
[ ALERTA ]: Activando protocolo de emergencia industrial!
[ ALERTA ]: Sirenas y balizas encendidas.

--- FIN DE LA SIMULACION ---
